# Demo — Onde o vetor mora: Azure Cosmos DB for NoSQL

**Objetivo:** guardar cada trecho do acervo **junto com o seu vetor**, no mesmo item de um contêiner do Azure Cosmos DB for NoSQL, buscar por similaridade em SQL (`VectorDistance`) e fechar o RAG. Ilustra os slides 20 a 23 ("Onde o vetor mora"): a linha **Cosmos DB for NoSQL** da tabela do slide 22 e onde o DiskANN do slide 23 entraria (aqui o índice é `quantizedFlat`).

1. **Preparar os vetores** → os mesmos trechos e o mesmo contrato de embedding dos outros notebooks;
2. **Criar a estrutura vetorial** → banco, contêiner, política vetorial e índice `quantizedFlat`;
3. **Gravar os trechos** → `upsert` pelo `id` do trecho, com o custo de cada gravação em RUs;
4. **Buscar por similaridade** → `VectorDistance` em SQL, com duas perguntas do gabarito;
5. **RAG completo** → a resposta com citação, a partir dos 3 primeiros trechos.

> **Cenário 100% fictício.** A Contoso Industrial e todos os documentos, projetos, pessoas e números do `acervo/` existem apenas para fins didáticos.

**Pré-requisito:** uma conta do **Azure Cosmos DB for NoSQL**, provisionada ou serverless, com o recurso **Vector Search for NoSQL API** habilitado. A aprovação é automática, mas pode levar até 15 minutos para valer.

- **No portal (prefira este caminho):** *Settings* → *Features* → *Vector Search for NoSQL API* → *Enable*.
- **Pela CLI:** `az cosmosdb update --resource-group <grupo> --name <conta> --capabilities EnableNoSQLVectorSearch`. Cuidado: `--capabilities` substitui a lista **inteira** de capacidades da conta. Confira antes com `az cosmosdb show --resource-group <grupo> --name <conta> --query capabilities` e repita as que já existem. Se a lista trouxer `EnableServerless`, por exemplo: `--capabilities EnableServerless EnableNoSQLVectorSearch`.

O notebook cria o banco `cap05` **sem throughput compartilhado**, porque a busca vetorial não é suportada nesse modo, e dentro dele o contêiner `trechos`.

## 0. Preparação

Variáveis necessárias:

- `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_DEPLOYMENT` (deployment de chat, usado na Execução 5)
- `AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, `AZURE_OPENAI_EMBEDDING_MODEL` e `AZURE_OPENAI_EMBEDDING_DIMENSIONS`: os mesmos valores dos outros notebooks
- `AZURE_COSMOS_ENDPOINT`, no formato `https://<conta>.documents.azure.com:443/`, e `AZURE_COSMOS_KEY`, a chave primária da conta (em *Keys*, no portal): é ela que permite ao notebook criar o banco e o contêiner

Os valores vêm do arquivo `Cap05/.env` (copie o `.env.example` e preencha); o que faltar é perguntado na hora da execução.

> Não grave chaves diretamente no notebook. Em produção, prefira Microsoft Entra ID à chave da conta (veja a leitura arquitetural da Execução 5).

In [ ]:
%pip install -q "openai>=2" "python-dotenv>=1.0" "pandas>=2.1" "azure-cosmos>=4.17"

In [ ]:
import os
from getpass import getpass
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))  # lê o .env da pasta Cap05, se existir

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or input(
    "AZURE_OPENAI_ENDPOINT (ex.: https://meu-recurso.openai.azure.com): "
).strip()
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY") or getpass("AZURE_OPENAI_API_KEY: ")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT") or input(
    "AZURE_OPENAI_DEPLOYMENT (deployment de chat): "
).strip()
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT") or input(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT (deployment de embedding): "
).strip()

# O modelo e a dimensão do embedding são o contrato entre indexação e consulta.
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# Pasta Cap05: a primeira pasta, subindo a partir daqui, que contém o acervo.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "acervo").is_dir())
PASTA_ACERVO = RAIZ / "acervo"

print("Cliente configurado.")
print("Deployment de chat:", AZURE_OPENAI_DEPLOYMENT)
print("Embedding:", AZURE_OPENAI_EMBEDDING_DEPLOYMENT, f"({EMBEDDING_MODEL}, {EMBEDDING_DIMENSIONS} dimensões)")
print("Acervo:", PASTA_ACERVO)

In [ ]:
from azure.cosmos import CosmosClient

AZURE_COSMOS_ENDPOINT = os.getenv("AZURE_COSMOS_ENDPOINT") or input(
    "AZURE_COSMOS_ENDPOINT (ex.: https://minha-conta.documents.azure.com:443/): "
).strip()
AZURE_COSMOS_KEY = os.getenv("AZURE_COSMOS_KEY") or getpass("AZURE_COSMOS_KEY (chave primária da conta): ")

NOME_BANCO = "cap05"
NOME_CONTEINER = "trechos"
cliente_cosmos = CosmosClient(AZURE_COSMOS_ENDPOINT, credential=AZURE_COSMOS_KEY)

print("Cosmos DB:", AZURE_COSMOS_ENDPOINT, "| banco:", NOME_BANCO, "| contêiner:", NOME_CONTEINER)

### Funções auxiliares

- `carregar_trechos`, `texto_para_embedding` e `gerar_embeddings`: as mesmas do notebook `tipos-de-rag/00`. Um trecho por seção `## `, e cada trecho vira um vetor com o mesmo modelo e a mesma dimensão.
- `PERGUNTAS` e `posicao`: o gabarito (`acervo/perguntas.json`) diz qual trecho responde cada pergunta; `posicao` diz em que lugar ele apareceu.
- `responder`: a geração com citação, usada na Execução 5.

In [ ]:
import re


def carregar_trechos(pasta_acervo: Path) -> list[dict]:
    """Lê os .md do acervo e devolve um trecho por seção "## " (chunking pela estrutura do documento)."""
    trechos = []
    for caminho in sorted(pasta_acervo.rglob("*.md")):
        fonte = caminho.relative_to(pasta_acervo).as_posix()
        doc_id = fonte.removesuffix(".md").replace("/", "_")
        texto = caminho.read_text(encoding="utf-8")
        titulo = texto.splitlines()[0].lstrip("# ").strip()
        # O que vem antes do primeiro "## " é o cabeçalho do documento (título e metadados).
        for numero, secao in enumerate(re.split(r"(?m)^## ", texto)[1:], start=1):
            nome_secao, _, conteudo = secao.partition("\n")
            trechos.append({
                "id": f"{doc_id}_{numero:02d}",
                "doc_id": doc_id,
                "titulo": titulo,
                "secao": nome_secao.strip(),
                "conteudo": conteudo.strip(),
                "fonte": fonte,
            })
    return trechos


def texto_para_embedding(trecho: dict) -> str:
    """O texto que vira vetor: título do documento, seção e conteúdo."""
    return f"{trecho['titulo']} — {trecho['secao']}\n{trecho['conteudo']}"


def gerar_embeddings(textos: list[str], lote: int = 16) -> list[list[float]]:
    vetores = []
    for i in range(0, len(textos), lote):
        resposta = client.embeddings.create(
            model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
            input=textos[i:i + lote],
            dimensions=EMBEDDING_DIMENSIONS,
        )
        vetores.extend(item.embedding for item in resposta.data)
    return vetores

In [ ]:
import json

pd.set_option("display.max_colwidth", None)  # tabelas mostram o texto inteiro

PERGUNTAS = {p["id"]: p for p in json.loads((PASTA_ACERVO / "perguntas.json").read_text(encoding="utf-8"))}


def posicao(ids_retornados: list[str], grupo: list[str]) -> int | None:
    """Posição (1, 2, 3...) do primeiro trecho esperado na lista retornada, ou None se ele não apareceu."""
    for i, id_trecho in enumerate(ids_retornados, start=1):
        if id_trecho in grupo:
            return i
    return None


INSTRUCOES_RAG = """
Responda em português, exclusivamente com base nos trechos fornecidos.
Cite a fonte de cada afirmação no formato [id do trecho].
Se os trechos não trouxerem a informação, responda:
'Não há evidência suficiente nos trechos recuperados.'
""".strip()


def responder(pergunta: str, trechos_recuperados: list[dict]) -> str:
    contexto = "\n\n".join(
        f"[{t['id']}] {t['titulo']} — {t['secao']}\n{t['conteudo']}" for t in trechos_recuperados
    )
    resposta = client.responses.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        instructions=INSTRUCOES_RAG,
        input=f"TRECHOS RECUPERADOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        store=False,
    )
    return resposta.output_text

---

# Execução 1 — Preparar os vetores

Os mesmos trechos do índice compartilhado (`tipos-de-rag/00`), vetorizados com o mesmo contrato de embedding (`EMBEDDING_MODEL` e `EMBEDDING_DIMENSIONS`): o chunking e o embedding não dependem de onde o vetor vai morar. Trocar de base é migração, mas o conteúdo se aproveita (slide 10); trocar o modelo de embedding obriga a refazer todos os vetores.

In [ ]:
trechos = carregar_trechos(PASTA_ACERVO)
vetores = gerar_embeddings([texto_para_embedding(t) for t in trechos])

print(f"{len(trechos)} trechos de {len({t['doc_id'] for t in trechos})} documentos")
print(f"{len(vetores)} vetores, cada um com {len(vetores[0])} dimensões")

---

# Execução 2 — Criar a estrutura vetorial

No Cosmos DB, o vetor é **uma propriedade do item**, ao lado do texto e dos metadados. A estrutura vetorial tem três partes:

- **Banco `cap05` e contêiner `trechos`**, com chave de partição `/doc_id`: os trechos de um mesmo documento ficam na mesma partição lógica.
- **Política vetorial** (`vector_embedding_policy`): o caminho `/embedding`, o tipo `float32`, a função `cosine` e a dimensão `EMBEDDING_DIMENSIONS`, o mesmo contrato do `.env`.
- **Política de indexação** (`indexing_policy`): um índice vetorial `quantizedFlat` em `/embedding`, e `/embedding/*` fora do índice comum. Indexar cada número do vetor como uma propriedade qualquer deixaria toda gravação mais cara e mais lenta.

### Os limites desta base (slide 22)

| Serviço | Dimensões | Índices | Reranking | Quando é a escolha certa |
|---|---|---|---|---|
| Cosmos DB for NoSQL | 505 a 4.096 | Flat, quantizedFlat, DiskANN | Sim | Dado operacional e vetor juntos, com atualização contínua |

Os "505 a 4.096" dependem do tipo de índice vetorial (Microsoft Learn):

| Índice | Até quantas dimensões | Como busca | Quando usar |
|---|---|---|---|
| `flat` | 505 | Força bruta, recall de 100% | Vetores pequenos; não cabe com as 1.536 dimensões do `.env.example` |
| `quantizedFlat` | 4.096 | Força bruta sobre vetores comprimidos; recall um pouco abaixo de 100% | Até ~50 mil vetores no escopo da busca: é o caso deste acervo |
| `diskANN` | 4.096 | Grafo DiskANN, busca aproximada | Acima de ~50 mil vetores |

### O algoritmo do índice (slide 23)

- O `diskANN` é o DiskANN do slide 23: recall alto, memória baixa e lida bem com dado que muda, o que combina com a atualização contínua da linha do slide 22. Ele entraria aqui se a busca passasse de ~50 mil vetores; com os 50 trechos do acervo, o `quantizedFlat` basta.
- O rodapé do slide 23 vale para os três: *"O trade-off é sempre recall, memória e tempo de indexação."* O `flat` e o `quantizedFlat` buscam por força bruta: comparam a pergunta com todos os vetores. O `flat` usa os vetores sem compressão e garante o recall; o `quantizedFlat` usa os vetores comprimidos e, por causa da compressão, cede um pouco de recall em troca de menos latência e menos RUs. Só o `diskANN` faz busca aproximada: não compara com todos e não garante 100% de recall, mas mantém o recall alto, com latência e custo em RUs baixos.

> `quantizedFlat` e `diskANN` só entram em ação a partir de **1.000 vetores**. Com os 50 trechos deste acervo, o Cosmos DB faz uma varredura completa, e a busca pode custar mais RUs. E a política vetorial não se altera no lugar: para mudar dimensão, função ou tipo de índice, o caminho seguro é recriar o contêiner (o Learn também descreve remover e recriar o caminho vetorial).

In [ ]:
from azure.cosmos import PartitionKey

politica_vetorial = {
    "vectorEmbeddings": [
        {"path": "/embedding", "dataType": "float32", "distanceFunction": "cosine", "dimensions": EMBEDDING_DIMENSIONS},
    ],
}
politica_indexacao = {
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": "/embedding/*"}],  # o vetor fica fora do índice comum
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
}

banco = cliente_cosmos.create_database_if_not_exists(id=NOME_BANCO)  # sem throughput compartilhado
conteiner = banco.create_container_if_not_exists(
    id=NOME_CONTEINER,
    partition_key=PartitionKey(path="/doc_id"),
    indexing_policy=politica_indexacao,
    vector_embedding_policy=politica_vetorial,
)

# Se o contêiner já existia, ele volta como estava: create_container_if_not_exists não altera nada.
propriedades = conteiner.read()
politica_registrada = propriedades.get("vectorEmbeddingPolicy", {}).get("vectorEmbeddings", [])
dimensoes_registradas = next((v.get("dimensions") for v in politica_registrada if v.get("path") == "/embedding"), None)
if dimensoes_registradas != EMBEDDING_DIMENSIONS:
    raise RuntimeError(
        f"O contêiner '{NOME_CONTEINER}' já existe com outra política vetorial ({politica_registrada}), "
        f"e o .env pede {EMBEDDING_DIMENSIONS} dimensões. A política vetorial não se altera no lugar: "
        "rode a limpeza no fim do notebook (APAGAR_RECURSOS = True), que apaga o contêiner, "
        "e volte a flag para False antes de executar tudo de novo."
    )

print(f"Contêiner '{NOME_CONTEINER}' pronto, com chave de partição {propriedades['partitionKey']['paths'][0]}.")
print("Política vetorial e índice vetorial, como o serviço registrou:")
display(pd.DataFrame(politica_registrada).merge(pd.DataFrame(propriedades["indexingPolicy"]["vectorIndexes"]), on="path"))

---

# Execução 3 — Gravar os trechos

Cada trecho vira **um item**: texto, metadados e vetor no mesmo JSON. `upsert_item` grava pelo `id` do trecho, que é estável (documento + número da seção): rodar o notebook de novo **substitui** os mesmos itens, em vez de duplicar. A chave de partição sai do próprio item (`doc_id`).

Toda operação no Cosmos DB custa **Request Units (RUs)**, e a resposta de cada gravação traz o custo no cabeçalho `x-ms-request-charge`. O Microsoft Learn estima ~35 RUs para inserir um item com vetor de 1.536 dimensões `float32` e índice `quantizedFlat`: é uma ordem de grandeza, não um preço.

> Numa conta provisionada com pouco RU/s (o mínimo do throughput manual é 400 RU/s), a carga pode passar do limite por segundo. O serviço responde 429, e o SDK espera e tenta de novo sozinho (até 9 vezes, por padrão): a célula só demora um pouco mais.

In [ ]:
custo_ru = 0.0
for trecho, vetor in zip(trechos, vetores):
    resposta = conteiner.upsert_item({**trecho, "embedding": vetor})
    custo_ru += float(resposta.get_response_headers()["x-ms-request-charge"])

total_no_conteiner = list(conteiner.query_items(query="SELECT VALUE COUNT(1) FROM c", enable_cross_partition_query=True))[0]

print(f"Gravados: {len(trechos)} trechos | custo: {custo_ru:.0f} RUs, ou {custo_ru / len(trechos):.1f} RUs por trecho")
print(f"Itens no contêiner: {total_no_conteiner} (trechos no acervo: {len(trechos)})")
if total_no_conteiner > len(trechos):
    print("Há itens a mais: sobras de uma carga anterior. upsert_item não apaga nada.")

---

# Execução 4 — Buscar por similaridade

A busca é uma consulta SQL: `VectorDistance` compara o vetor da pergunta com o `embedding` de cada item, `ORDER BY VectorDistance(...)` ordena do mais parecido para o menos parecido e `TOP @k` corta a lista. A pergunta vira vetor **no cliente**, com o mesmo `gerar_embeddings` da carga, e entra na consulta como parâmetro (`@vetor`), nunca colada no texto do SQL.

- Com `cosine`, o `VectorDistance` já devolve a **similaridade** de cosseno, de −1 a 1: quanto maior, mais parecido. Não há conversão a fazer. No AI Search (notebook `tipos-de-rag/01`), o score era `1 / (1 + distância)`. O nome engana: com `euclidean`, o valor seria uma distância, e menor seria melhor.
- `TOP` é obrigatório na prática: sem ele, o Learn avisa que a busca vetorial gasta mais RUs e demora mais, e o SDK Python recusa a consulta quando ela atravessa várias partições físicas.
- Não ponha `ASC` nem `DESC` no `ORDER BY VectorDistance(...)`: a ordem já vem do mais parecido para o menos parecido, e com `ASC` o serviço recusa a consulta.
- A chave de partição é `/doc_id`, e a pergunta não sabe de que documento virá a resposta: a busca atravessa as partições (`enable_cross_partition_query=True`).

### O que esperar e por que pode variar

- Na pergunta em linguagem natural, o trecho certo tende a vir no topo, como no AI Search.
- No SKU 48213-B, os três SKUs tendem a aparecer quase empatados: a troca de base não muda a natureza do vetor (slide 14). Pode dar sorte, e o 48213-B vir em 1º; o que importa é a diferença para o 2º.
- Os números dependem do modelo e da dimensão do embedding. Com menos de 1.000 vetores, o Cosmos DB compara a pergunta com todos os itens; a partir daí, o `quantizedFlat` compara vetores comprimidos, e vizinhos quase empatados podem trocar de lugar.

In [ ]:
CONSULTA_VETORIAL = """
SELECT TOP @k c.id, c.titulo, c.secao, c.conteudo, c.fonte,
    VectorDistance(c.embedding, @vetor) AS similaridade
FROM c
ORDER BY VectorDistance(c.embedding, @vetor)
"""


def buscar(pergunta: str, k: int = 5) -> list[dict]:
    """Busca vetorial no Cosmos DB: vetoriza a pergunta no cliente e pede os k itens de vetor mais parecido."""
    vetor = gerar_embeddings([pergunta])[0]
    return list(conteiner.query_items(
        query=CONSULTA_VETORIAL,
        parameters=[{"name": "@k", "value": k}, {"name": "@vetor", "value": vetor}],
        enable_cross_partition_query=True,
    ))


resultados = {pid: buscar(PERGUNTAS[pid]["pergunta"], k=5) for pid in ["linguagem-natural", "string-exata"]}

for pid, top5 in resultados.items():
    esperados = PERGUNTAS[pid]["trechos_esperados"][0]
    lugar = posicao([t["id"] for t in top5], esperados)
    situacao = f"{lugar}º" if lugar else "fora do top 5"
    display(Markdown(f"**{PERGUNTAS[pid]['pergunta']}** · posição do trecho certo: {situacao}"))
    display(pd.DataFrame([
        {
            "posição": i,
            "similaridade": round(t["similaridade"], 4),
            "diferença para o 1º": round(t["similaridade"] - top5[0]["similaridade"], 4),
            "trecho": f"{t['titulo']} — {t['secao']}",
            "é o trecho certo?": "sim" if t["id"] in esperados else "",
        }
        for i, t in enumerate(top5, start=1)
    ]))

### O que observar

- Na pergunta em linguagem natural, compare com o notebook `tipos-de-rag/01`: a ordem tende a ser a mesma, só a escala do número muda (similaridade de cosseno aqui, `1 / (1 + distância)` no AI Search).
- No SKU 48213-B, olhe a coluna **diferença para o 1º**: os irmãos 48213-C e 48213-D continuam colados. Trocar a base não muda a natureza do vetor: o quase empate do slide 14 está aqui também.

### Leitura arquitetural

- Busca híbrida e reranking são decisões de **recuperação** (os notebooks de `tipos-de-rag`); a base é uma decisão de **operação** (slide 21). Guardar o vetor no Cosmos DB não faz o SKU bater a string.

---

# Execução 5 — RAG completo

Os 3 primeiros trechos da busca vão para o modelo, que responde citando o `id` de cada trecho usado. É a mesma geração das outras bases da série e de `tipos-de-rag`: o modelo não sabe em que base o trecho estava guardado.

In [ ]:
pid = "linguagem-natural"
pergunta = PERGUNTAS[pid]["pergunta"]
top3 = resultados[pid][:3]
lugar = posicao([t["id"] for t in top3], PERGUNTAS[pid]["trechos_esperados"][0])
enviados = ", ".join(f"`{t['id']}`" for t in top3)
situacao = f"sim, em {lugar}º lugar" if lugar else "não"

display(Markdown(f"### {pergunta}"))
display(Markdown(f"**Trechos enviados ao modelo:** {enviados}  \n**O trecho certo está entre eles?** {situacao}"))
display(Markdown(responder(pergunta, top3)))

### O que observar

- Leia a linha **O trecho certo está entre eles?** antes de julgar a resposta: ela separa erro de recuperação de erro de geração (slide 29).

### Leitura arquitetural

- Slide 22, linha do Cosmos DB, coluna **Quando é a escolha certa**: *"Dado operacional e vetor juntos, com atualização contínua"*. É o "sim" à pergunta do slide 21, **o dado muda em tempo real?**: o trecho e o vetor estão no mesmo item, e a aplicação que atualiza o texto regrava o vetor na mesma operação, sem um índice separado para sincronizar.
- **Quem opera:** o time que já sustenta o Cosmos DB da aplicação (conta, chave de partição, política de indexação). **Quem monitora:** o mesmo time, no Azure Monitor: RUs consumidas e respostas 429. **Quem paga:** RU e armazenamento. Na conta provisionada, o RU/s reservado (usado ou não) mais os GB de dados e índices; na serverless, as RUs consumidas mais os mesmos GB.
- **Híbrido no Cosmos DB** é uma consulta: `ORDER BY RANK RRF(VectorDistance(...), FullTextScore(...))`. Pede política e índice de texto completo no contêiner, e o texto completo em pt-BR ainda está em preview. O **reranking** do slide 22 é o Semantic Reranker, também em preview, só com Microsoft Entra ID e até 50 documentos por chamada.
- **Chave × Entra ID:** a chave da conta pode tudo, e por isso este notebook consegue criar banco e contêiner. Em produção, use Microsoft Entra ID e desligue as chaves (`disableLocalAuth`): a identidade da aplicação recebe um papel de dados (por exemplo, *Cosmos DB Built-in Data Contributor*) para gravar e consultar itens, e banco e contêiner passam a ser criados pelo plano de controle (portal, Bicep ou CLI).

### Fala sugerida

> “A pergunta não é qual banco vetorial é o melhor. É se o dado muda em tempo real.”

---

# Quadro final — o que muda ao trocar a base?

| Execução | O que vimos | Decisão arquitetural |
|---|---|---|
| 1 | Os mesmos trechos e o mesmo contrato de embedding dos outros notebooks | O contrato de embedding não depende da base |
| 2 | Política vetorial e índice `quantizedFlat`, como o serviço registrou | Tipo de índice pelo volume de vetores; a política não se altera no lugar, então decida antes |
| 3 | Upsert pelo `id`, sem duplicar, com o custo de cada gravação em RUs | Texto, metadados e vetor no mesmo item; cada gravação tem preço |
| 4 | `VectorDistance` já devolve a similaridade, e o SKU continua quase empatado | A base não muda a natureza do vetor; híbrido e reranking são outra decisão |
| 5 | A resposta com citação é a mesma dos outros notebooks | O modelo não sabe onde o vetor mora: a base se escolhe pela operação |

## Frase de fechamento

> **Escolher a base é decisão de operação, não de benchmark: cada componente novo precisa de alguém para operar, monitorar e pagar.**

---

# Referências oficiais

- Azure Cosmos DB for NoSQL — busca vetorial: recurso, políticas, tipos de índice e limites:  
  https://learn.microsoft.com/azure/cosmos-db/vector-search

- Azure Cosmos DB for NoSQL — indexar e consultar vetores em Python:  
  https://learn.microsoft.com/azure/cosmos-db/how-to-python-vector-index-query

- Azure Cosmos DB — função VectorDistance:  
  https://learn.microsoft.com/cosmos-db/query/vectordistance

- Azure Cosmos DB — desempenho e custo (RUs) da busca vetorial:  
  https://learn.microsoft.com/azure/cosmos-db/gen-ai/vector-search-performance-tips

---

# Limpeza (opcional)

`APAGAR_RECURSOS = True` apaga o contêiner `trechos`, que este notebook criou. O banco `cap05` só é apagado se ficar vazio: se ele tiver outros contêineres, fica onde está, com eles. Numa conta provisionada, o contêiner cobra o RU/s reservado mesmo sem uso: apague no fim da aula.

Mudou a dimensão do embedding no `.env`? Reinicie o kernel, porque os notebooks leem o `.env` só uma vez. Com a dimensão nova, a Execução 2 para com um erro, porque a política vetorial do contêiner não se altera no lugar. Apague o contêiner aqui e rode o notebook inteiro de novo, com `APAGAR_RECURSOS` de volta em `False`.

Mudou só o modelo, com a mesma dimensão? Nada falha: a política vetorial guarda a dimensão, não o modelo. Reinicie o kernel e rode o notebook inteiro, para regravar os 50 vetores com o modelo novo (slide 10: vetores de modelos diferentes não se comparam).

In [ ]:
APAGAR_RECURSOS = False  # True apaga o contêiner trechos (e o banco cap05, se ele ficar vazio)

if APAGAR_RECURSOS:
    banco = cliente_cosmos.get_database_client(NOME_BANCO)
    banco.delete_container(NOME_CONTEINER)
    print(f"Contêiner '{NOME_CONTEINER}' apagado.")
    outros = [c["id"] for c in banco.list_containers()]
    if outros:
        print(f"O banco '{NOME_BANCO}' foi mantido, porque tem outros contêineres: {', '.join(outros)}.")
    else:
        cliente_cosmos.delete_database(NOME_BANCO)
        print(f"O banco '{NOME_BANCO}' ficou vazio e também foi apagado.")
else:
    print(f"Nada foi apagado. O contêiner '{NOME_CONTEINER}' continua no banco '{NOME_BANCO}'.")